### Importing libraries

In [1]:
# # %pip install ddgs trafilatura -q
# %pip install openai-agents 

In [2]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
from openai import OpenAI
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool



In [14]:
# JUDGE_MODEL = "gpt-4.1-mini"
MODEL = "gpt-4.1-mini"

### Setup

In [4]:
# api
load_dotenv()

OpenAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key= OpenAI_API_KEY)

if OpenAI_API_KEY is None:
    raise Exception ("No API Key for Open AI")
else:
    print(OpenAI_API_KEY[:8])

sk-proj-


#### Step 1: Define Tools

In [5]:
#search web tool

@function_tool
def search_web(query:str):
    """Seach the web using ddgs browser. Return the top 3 results"""
    #initialize a ddgs object:
    ddgs = DDGS()
    results = ddgs.text(query, max_results = 3)
    print(f'\u2705 Got results')
    #returning in json bc LLM doesnt work with dictionaries, so json converts to text
    return json.dumps(results, indent= 2)

In [6]:
#Fetching the information from the search results. Scraping the url
@function_tool
def fetch_url(url:str):
    """Extracting text from fetched from web result url"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f'\u2705 Got Text {len(text)} characters')
            return text
        else:
            print(f'\u274C Failed to fetch or extract text.')
    return f'Could not extract text from {url}. Try a different source.'

### Step 2: System Prompt

#### This tells the LLM who it is and how to behave. The key things:
* - What its job is
* - What tools it has
* - What process to follow
* - What output format to produe

In [7]:

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: DO NOT just tool call over and over again. You need to THINK before doing a tool call. This is not a comprehensive brief if you do not think before tool calls. I want to see you working trhough deciding what are the
best sources or urls to use. You NEED AT LEAST 6 sources. Make sure to think through which is the best 6 of all your sources. Try and think between tool calls to determine if you need to search the web more for a better topic search
or fetch a different url. You should use your discernment to try and think through the research brief to create a concise 2 paragraph brief, with cited sources. 

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""



### Step 3: Define the Agent

In [15]:
#This is used instead of the def run_research_agent loop we usually make:
research_agent = Agent(
    name = "Research Agent",
    instructions= RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_url]
)

### Let's Run it!

In [16]:
#research agent defined above and runner runs the runner
result = await Runner.run(
    research_agent,
    input = "Research the following topic and produce a comprehensive research brief: How AI is used in Helathcare",
    max_turns= 30
)

✅ Got results
✅ Got Text 109439 characters
✅ Got Text 35740 characters
✅ Got results
✅ Got Text 25909 characters
✅ Got Text 32340 characters
✅ Got Text 8301 characters


In [17]:
print(f'Agent: {result.last_agent.name}')
print(f'-----')
display(Markdown(result.final_output))

Agent: Research Agent
-----


I have gathered information from multiple reputable sources, including recent academic reviews, industry reports, and expert discussions on how AI is used in healthcare. Here is a comprehensive research brief:

---

**Research Brief: How AI is Used in Healthcare**

Artificial Intelligence (AI) is fundamentally transforming healthcare across clinical, operational, research, and patient-engagement domains. At its core, AI leverages machine learning, deep learning, natural language processing, and generative models to analyze vast and diverse medical data, including electronic health records (EHRs), medical imaging, genomic profiles, and real-world health data, to improve diagnostics, treatment, and healthcare delivery efficiency.

Key AI applications include enhanced medical imaging diagnostics that achieve or surpass human expert accuracy in detecting cancers, fractures, and other conditions. AI-driven clinical decision support systems analyze patient histories and labs to flag risks and suggest treatment options. AI-based drug discovery accelerates compound screening and protein structure prediction, reducing research timelines significantly. Robotics powered by AI improve precision in surgery and rehabilitation. AI-enhanced telemedicine, virtual health assistants, and wearable devices facilitate continuous patient monitoring, personalized health guidance, and better chronic disease management. Healthcare operational AI (agentic AI) automates administrative tasks such as prior authorization, claims processing, and scheduling, easing workforce shortages and administrative burdens. Moreover, AI is poised to drive precision medicine through integration of multi-omics data and digital twins, allowing tailored, dynamic treatment planning.

Despite rapid adoption—with 70% of surveyed healthcare organizations already using AI—the field faces challenges. These include ensuring patient data privacy and security, managing regulatory and ethical considerations, mitigating biases inherent in training data, ensuring model interpretability for clinical trust, and aligning AI system deployment with healthcare workflows and human oversight. Hurdles also exist around clinical validation, public trust, and equitable access, particularly in low-resource settings.

Future advancements are expected in AI agents capable of autonomous clinical workflow management, deeper integration with digital and genomic data, and the broader use of AI in resource-limited environments to reduce healthcare disparities. To realize AI's full potential, collaborative efforts among technology developers, healthcare providers, policymakers, and patients are essential, focusing on ethical, sustainable, and transparent AI integration.

---

**Key Facts and Statistics**

- 70% of healthcare organizations are actively using AI, particularly in clinical decision support, medical imaging, and workflow optimization (NVIDIA report, 2026).
- AI algorithms can detect diabetic retinopathy with over 87% sensitivity and 90% specificity, having FDA approval for autonomous use.
- AI can predict sepsis onset hours earlier than traditional methods, improving emergency care outcomes.
- AI accelerates early-stage drug discovery by over 70%, significantly shortening timelines.
- More than half of surveyed physicians use or are interested in using AI tools, primarily for reducing administrative burden.
- Administrative functions consume 15-30% of U.S. healthcare spending, with AI automating tasks like prior authorizations that occupy up to 13 hours weekly per physician.
- Limitations include occasional AI hallucinations (misinformation), lack of contextual reasoning, and decreased public trust (down from 52% openness in 2024 to 42% in 2026).

---

**Main Themes and Arguments**

- AI is not a replacement for clinicians but a tool to augment decision-making and operational efficiency.
- Human-centered design and integration with clinical workflows are crucial for AI adoption and trust.
- Transparency, explainability (XAI), and regulatory oversight ensure safe AI use.
- Bias mitigation through diverse data and rigorous evaluation is necessary to avoid exacerbating healthcare disparities.
- AI shows immense potential in transforming personalized medicine, predictive analytics, and pharmaceutical workflows.
- Ethical deployment demands safeguarding patient privacy and clear allocation of liability.
- Sustained clinician training and patient education on AI capabilities and limitations are essential.
- Challenges include scalability, real-time data processing, interoperability, and public acceptance.

---

**Notable Data Points**

- OpenAI's ChatGPT Health and Anthropic's Claude for Healthcare integrate with medical records and wellness data to offer personalized guidance.
- AI-driven autonomous prior authorization systems reduce physician paperwork by assembling submission packages and tracking approvals.
- AI-powered surgical robotics improve minimally invasive procedure precision and reduce recovery times.
- AI-enabled wearable devices provide continuous monitoring of vitals such as heart rhythms and glucose levels, enabling proactive care.
- AI in imaging (e.g., Google DeepMind for eye diseases) delivers diagnostic accuracy on par with top specialists.
- Agentic AI solutions carrying out multistep workflows have shown promise but also highlight the need for robust human oversight and governance.

---

**Source URLs**

1. https://pmc.ncbi.nlm.nih.gov/articles/PMC12455834/  
2. https://pmc.ncbi.nlm.nih.gov/articles/PMC8285156/  
3. https://ep.jhu.edu/news/ai-in-healthcare-applications-and-impact/  
4. https://higoodie.com/blog/ai-in-healthcare-2026/  
5. https://helpware.com/blog/agentic-ai-healthcare  
6. https://www.linkedin.com/pulse/we-take-our-whiskey-neat-key-insights-from-ai-hans-mulder-a3e0c  

---

This brief synthesizes foundational understanding, current applications, benefits, and challenges, as well as emerging trends in AI use across the healthcare ecosystem.